# Stylistic Feature-Union Architecture — WELFake 98.85%

**Ties the published state of the art using linear models on two CPU cores.**

---

## The idea, and where it came from

This architecture was predicted by a diagnostic, not found by trying bigger models.

An earlier **shortcut audit** of WELFake showed that a classifier reading **no words at all** —
only punctuation rates, capitalisation and length — reaches macro-F1 **0.739 on punctuation
alone** and **0.804 with all such cues combined**, against a full model's 0.959. Most of
WELFake's discriminative signal is therefore **stylistic**: casing, punctuation, spacing, register.

But `TfidfVectorizer(analyzer="word")` **lower-cases its input and strips punctuation during
tokenisation**. Every model we had built was blind to the corpus's strongest cue. A recurrent
network did not fix this either — an attention BiLSTM trained from scratch reached only 96.30%,
because it had to rediscover that signal through a 128-d embedding.

The fix is to hand the model the stylistic signal directly, in three blocks:

| Block | Captures | Key setting |
|---|---|---|
| 1. word TF-IDF (1–2 grams, 100k) | topical / lexical content | lower-cased (standard) |
| 2. **char_wb TF-IDF (3–4 grams, 80k)** | **casing, punctuation, affixes, boilerplate** | **`lowercase=False`** |
| 3. 12 surface statistics | length, caps, punctuation rates, register | `StandardScaler` |

Three linear models are fitted on the union and combined by a stacked logistic regression.

**Result:** 98.85% on the published protocol (ties SOTA), 98.57% on a stricter de-duplicated one.

## 1. Setup

In [1]:
import re, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
FAKE, REAL = 0, 1                 # repo-wide convention
WELFAKE_CSV = Path("/mnt/user-data/uploads/Downloads/WELFake_Dataset.csv")
CHAR_CAP = 1200                   # characters fed to the char vectoriser
np.random.seed(SEED)
_STOP = frozenset(ENGLISH_STOP_WORDS)

## 2. Data, and two traps

**Trap 1 — duplicates.** WELFake contains 11.7% exactly duplicated articles. Removing them
*before* splitting is the honest protocol; published work generally keeps them, so the same
article can sit in both train and test. We run **both** protocols so the comparison with
published numbers is like-for-like and the leakage is measured rather than assumed.

**Trap 2 — inverted labels.** WELFake ships `0 = real, 1 = fake`, the opposite of FineFake and
ISOT. Verified directly: 21,266 of its label-0 articles carry a `(Reuters)` dateline against 16
of its label-1 articles. macro-F1 is symmetric so this does not change single-corpus scores, but
it silently destroys any cross-corpus work.

In [2]:
def load_welfake(deduplicate: bool) -> pd.DataFrame:
    df = pd.read_csv(WELFAKE_CSV)
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")
    df["doc"] = (df["title"] + ". " + df["text"]).str.strip()
    df = df[df["doc"].str.len() > 0].dropna(subset=["label"])
    if deduplicate:
        df = df.drop_duplicates(subset=["doc"])      # 72,134 -> 63,676
    df["label"] = 1 - df["label"].astype(int)        # flip onto fake=0, real=1
    return df.reset_index(drop=True)[["doc", "label"]]


DEDUP = False          # False = raw (published protocol); True = de-duplicated
df = load_welfake(DEDUP)
tr, tmp = train_test_split(df, test_size=.30, stratify=df["label"], random_state=SEED)
va, te  = train_test_split(tmp, test_size=.50, stratify=tmp["label"], random_state=SEED)
tr, va, te = (x.reset_index(drop=True) for x in (tr, va, te))
y_tr, y_va, y_te = (x["label"].to_numpy() for x in (tr, va, te))
print(f"rows {len(df):,}   train {len(tr):,}  val {len(va):,}  test {len(te):,}")

rows 72,134   train 50,493  val 10,820  test 10,821


## 3. Block 3 — the surface statistics

Twelve dense features, each a rate rather than a raw count so that document length does not
dominate. These duplicate part of what the character block captures, but as explicitly scaled
quantities a linear model can weight directly instead of aggregating over thousands of sparse
n-grams.

In [3]:
def surface_stats(texts: pd.Series) -> np.ndarray:
    '''The cues the shortcut audit showed carry most of WELFake's signal, made explicit.'''
    t = texts.astype(str)
    n_ch = t.str.len().clip(lower=1)
    toks = t.str.split()
    n_tk = toks.str.len().clip(lower=1)

    def stop_ratio(ws):
        return sum(w.lower() in _STOP for w in ws) / len(ws) if ws else 0.0

    return np.column_stack([
        np.log1p(n_tk),                                # 1  length in words
        np.log1p(n_ch),                                # 2  length in characters
        t.str.count(r"[A-Z]") / n_ch,                  # 3  capitalisation ratio
        t.str.count(r"[A-Z]{2,}") / n_tk,              # 4  ALL-CAPS runs
        t.str.count(r"!") / n_tk,                      # 5  exclamation rate
        t.str.count(r"\?") / n_tk,                     # 6  question rate
        t.str.count(r'["“”]') / n_tk,        # 7  quotation rate
        t.str.count(r"\.\.\.") / n_tk,                 # 8  ellipsis rate
        t.str.count(r"[0-9]") / n_ch,                  # 9  digit ratio
        t.str.count(r"\s") / n_ch,                     # 10 whitespace ratio
        t.str.count(r"[,;:]") / n_tk,                  # 11 mid-sentence punctuation
        toks.apply(stop_ratio),                        # 12 register proxy
    ]).astype(np.float64)

## 4. The feature union

`lowercase=False` on Block 2 is the single most important setting in the system. With it the
vectoriser treats `"BREAKING"` and `"Breaking"` as different features and encodes `!!!`, `?!`,
`...`, quote styles and spacing irregularities as ordinary n-grams. `char_wb` keeps n-grams
inside word boundaries, so the features read as affixes and punctuation clusters rather than
arbitrary character windows.

Truncating to 1,200 characters and `min_df=5` are tractability measures — character n-grams
generate an enormous number of non-zeros, and the untruncated version exhausted memory.

In [4]:
def build_features(tr_txt, va_txt, te_txt):
    # ---- Block 1: topical content. Lower-cased and punctuation-stripped by design.
    t0 = time.time()
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                           max_features=100_000, sublinear_tf=True,
                           strip_accents="unicode", dtype=np.float32)
    Wtr = word.fit_transform(tr_txt); Wva = word.transform(va_txt); Wte = word.transform(te_txt)
    print(f"word block {Wtr.shape}  ({time.time()-t0:.0f}s)")

    # ---- Block 2: THE decisive block. lowercase=False keeps casing AND punctuation.
    t0 = time.time()
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=5,
                           max_features=80_000, sublinear_tf=True,
                           lowercase=False, dtype=np.float32)
    Ctr = char.fit_transform(tr_txt.str.slice(0, CHAR_CAP))
    Cva = char.transform(va_txt.str.slice(0, CHAR_CAP))
    Cte = char.transform(te_txt.str.slice(0, CHAR_CAP))
    print(f"char block {Ctr.shape}  ({time.time()-t0:.0f}s)")

    # ---- Block 3: dense surface statistics, standardised on train
    sc = StandardScaler()
    Str = sc.fit_transform(surface_stats(tr_txt))
    Sva = sc.transform(surface_stats(va_txt)); Ste = sc.transform(surface_stats(te_txt))

    f32 = lambda M: sparse.csr_matrix(M, dtype=np.float32)
    return (sparse.hstack([Wtr, Ctr, f32(Str)], format="csr", dtype=np.float32),
            sparse.hstack([Wva, Cva, f32(Sva)], format="csr", dtype=np.float32),
            sparse.hstack([Wte, Cte, f32(Ste)], format="csr", dtype=np.float32),
            (Wtr, Wva, Wte))


Xtr, Xva, Xte, (Wtr, Wva, Wte) = build_features(tr["doc"], va["doc"], te["doc"])
print("union:", Xtr.shape)

word block (50493, 100000)  (80s)


char block (50493, 80000)  (49s)


union: (50493, 180012)


## 5. Metrics

macro-F1 is the headline; we also record accuracy, precision, recall, ROC-AUC, MCC and the two
operationally meaningful error rates — **fake_FNR** (fake articles let through) and **fake_FPR**
(real articles wrongly flagged).

The decision threshold is swept on **validation**, never on test, and the same sweep is applied
to every system so nothing wins by being tuned when others are not.

In [5]:
def tune_threshold(y, p):
    best_t, best_f = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        f = f1_score(y, np.where(p >= t, FAKE, REAL), average="macro")
        if f > best_f:
            best_f, best_t = f, t
    return best_t, best_f


def full_metrics(y, p, t) -> dict:
    pred = np.where(p >= t, FAKE, REAL)
    cm = confusion_matrix(y, pred, labels=[FAKE, REAL])
    tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    return {"macro_f1": f1_score(y, pred, average="macro"),
            "accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, pos_label=FAKE, zero_division=0),
            "recall": recall_score(y, pred, pos_label=FAKE, zero_division=0),
            "roc_auc": roc_auc_score((y == FAKE).astype(int), p),
            "mcc": matthews_corrcoef(y, pred),
            "fake_FNR": fn / max(fn + tp, 1),
            "fake_FPR": fp / max(fp + tn, 1)}


def p_fake(clf, X):
    return clf.predict_proba(X)[:, list(clf.classes_).index(FAKE)]

## 6. Models

Note the solver choice. `liblinear` (behind `LinearSVC`) copies the matrix to `float64`
internally and was **OOM-killed twice** on the 180k-feature union in a 7 GB container. The union
is therefore fitted with memory-light solvers only: SGD and lbfgs.

In [6]:
models = {
    "SGD (modified Huber)": SGDClassifier(loss="modified_huber", alpha=1e-6,
                                          class_weight="balanced", max_iter=3000,
                                          tol=1e-4, random_state=SEED),
    "SGD (log loss)":       SGDClassifier(loss="log_loss", alpha=1e-6,
                                          class_weight="balanced", max_iter=3000,
                                          tol=1e-4, random_state=SEED),
    "LogisticRegression":   LogisticRegression(max_iter=1500, C=4.0,
                                               class_weight="balanced", random_state=SEED),
}

results, P_va, P_te = [], {}, {}
for name, clf in models.items():
    t0 = time.time()
    clf.fit(Xtr, y_tr)
    pv, pt = p_fake(clf, Xva), p_fake(clf, Xte)
    P_va[name], P_te[name] = pv, pt
    t, _ = tune_threshold(y_va, pv)
    results.append({"model": name, **full_metrics(y_te, pt, t)})
    print(f"{name:22s} acc {results[-1]['accuracy']:.4f}  ({time.time()-t0:.0f}s)")

SGD (modified Huber)   acc 0.9860  (3s)


SGD (log loss)         acc 0.9705  (9s)


LogisticRegression     acc 0.9881  (40s)


## 7. Stacked combiner

A logistic regression over the three models' probabilities, fitted on validation. It adds only
about +0.04 points — essentially all of the gain comes from the features, not the ensembling.

In [7]:
Mva = np.column_stack(list(P_va.values()))
Mte = np.column_stack(list(P_te.values()))
stack = LogisticRegression(max_iter=2000, class_weight="balanced",
                           random_state=SEED).fit(Mva, y_va)
sv, st = p_fake(stack, Mva), p_fake(stack, Mte)
t_star, _ = tune_threshold(y_va, sv)
results.append({"model": "STACKED (3 models)", **full_metrics(y_te, st, t_star)})

res = pd.DataFrame(results).round(4)
res

,model,macro_f1,accuracy,precision,recall,roc_auc,mcc,fake_FNR,fake_FPR
0,SGD (modified Huber),0.9859,0.9860,0.9830,0.9898,0.9871,0.9719,0.0102,0.0181
1,SGD (log loss),0.9705,0.9705,0.9602,0.9835,0.9864,0.9412,0.0165,0.0432
2,LogisticRegression,0.9881,0.9881,0.9904,0.9863,0.9990,0.9761,0.0137,0.0101
3,STACKED (3 models),0.9885,0.9885,0.9896,0.9881,0.9990,0.9771,0.0119,0.0110


## 8. The ablation that matters

Fitting the **word block alone** isolates what the stylistic blocks actually bought.

In [8]:
ref = LogisticRegression(max_iter=1500, C=4.0, class_weight="balanced",
                         random_state=SEED).fit(Wtr, y_tr)
t, _ = tune_threshold(y_va, p_fake(ref, Wva))
word_only = full_metrics(y_te, p_fake(ref, Wte), t)

best = res.loc[res["accuracy"].idxmax()]
print(f"word block only           : {word_only['accuracy']:.4f}")
print(f"+ char_wb + surface stats : {best['accuracy']:.4f}"
      f"   ({best['accuracy'] - word_only['accuracy']:+.4f})")

word block only           : 0.9713
+ char_wb + surface stats : 0.9885   (+0.0172)


## 9. Where this sits

| Rank | Model | Accuracy | Protocol |
|---|---|---|---|
| 1 = | **This notebook — stacked** | **98.85%** | raw |
| 1 = | Hybrid CNN + dual BiLSTM (2025) | 98.85% | raw |
| 5 | **This notebook — stacked** | 98.57% | de-duplicated |
| 6 | BERT + BiLSTM | 98.10% | raw |
| 7 | BERT | 98.03% | raw |
| 8 | DistilBERT | 97.85% | raw |
| 11 | Attention BiLSTM | 97.66% | raw |
| 14 | SVM / WELFake original (2021) | 96.73% | raw |

We tie the published state of the art on its own protocol and beat every transformer result,
with linear models, no GPU, no pre-training, in about a minute of training.

### The honest caveat

This is a **better WELFake scorer, not demonstrably a better fake-news detector.** The
architecture wins by deliberately exploiting the stylistic regularity the audit identified as
substantially artifactual — character n-grams with casing preserved are ideal for capturing
*"this article came from a source that writes a certain way"*, which is source identification
rather than deception detection. Earlier transfer experiments showed models trained on one of
these corpora fall to near-chance on another, and a more artifact-sensitive model should if
anything transfer worse.

The leaderboard result and the audit are the same finding from two sides: the audit said WELFake
is largely solvable from style; this architecture solves it from style and reaches the top of the
table. That is evidence about the benchmark at least as much as about the model.

### Limitations

1. **Single seed** — treat the 98.85% as a tie, not a win.
2. **In-distribution only** — the transfer expectation above is a prediction, not a measurement.
3. **Validation reuse** — the combiner and the threshold share one validation split.
4. **Near-saturated benchmark** — sixteen systems inside a 3-point band on a corpus where
   content-free cues alone reach 80%.